# Are prices informative (Section 4)

Estimates how much uncertainty about each market's outcome is resolved by observing its price.
This notebook produces **Table 3** (information in Polymarket prices), **Table 4** (negative-risk
vs other markets) and the robustness numbers quoted in Section 4.3.

Contents:
1. The mutual information estimator
2. Point estimates: full sample (Table 3) and by market structure (Table 4)
3. Bootstrap standard errors

In [3]:
import pandas as pd
import numpy as np

from common import binary_entropy, insert_or_replace, nextpos
from bootstrap import *

In [4]:
panel = pd.read_parquet("../data/pipeline/panel.parquet")

## 1. The mutual information estimator

The information a price carries about the outcome $Y \in \{0, 1\}$ is the mutual information

$$I(Y;P) = H(Y) - H(Y \mid P),$$

the drop in outcome uncertainty from observing the price. `entropy_calculations` estimates it in four steps.

**Weights.** Markets trade for very different lengths of time, so counting every market-hour equally
would let long-lived markets dominate. By default (`weighting="market"`) each hour of market $i$
gets weight $1/T_i$, where $T_i$ is the number of hours it trades, so every market counts once.
`weighting="hour"` gives every market-hour weight 1, as a robustness check.

**Unconditional entropy $H(Y)$.** The weighted share of Yes outcomes $q$ gives
$H(Y) = -\big(q \log_2 q + (1-q)\log_2(1-q)\big)$, the uncertainty before looking at any price.

**Conditional entropy $H(Y \mid P)$.** Prices are grouped into 20 bins of width 0.05. Within each bin
we compute the entropy of the outcomes, then average across the price bins.

**Miller–Madow correction.** Entropy estimates are biased downward in finite samples, which
biases mutual information upward. The correction subtracts $(K-1)/(2N \ln 2)$ bits, where $K$ is the
number of price bins containing both outcomes and $N$ is the number of markets (our independent units).

The function returns the share of $H(Y)$ resolved by prices, $100 \cdot I(Y;P) / H(Y)$, which is the
headline number in the paper.

In [5]:
# another cell combining pipeline
def entropy_calculations(input, weighting="market", drop_last_hours = 0):
    market_entropy = binary_entropy(input["p"])

    insert_or_replace(input, nextpos(input, "p"), "market_entropy", market_entropy)
    # market entropy here isnt overall entropy (thats for the whole
    # distribution, a single number), its predictive uncertainty
    # at that point

    # robustness check
    # maybe information is inflated by the final hours of the market, where many times
    # the outcome is already known but the market hasn't closed yet
    ### this was a crude implementation since it removes actual trading hours too, 
    ### when the market is most informative. i will find a better way to implement this.
    if drop_last_hours > 0:
        df = input[input["time_to_resolution"] > drop_last_hours]
    else:
        df = input

    #### WEIGHTS
    # H(Y) and H(Y|P) have to be calculated on the same weights,
    # or else it doesnt make any sense to take the difference, it wont be MI either
    # we'll weight everything by 1/(hours in the market) so it doesnt skew towards
    # long markets

    if weighting == "market":
        # transform("size") counts the rows in each market and writes it onto every row of the market
        # so hours is the same length as df
        hours = df.groupby("market_id", observed=True)["yes_outcomes"].transform("size")
        w = 1/hours
    # robustness check where each market hour counts once
    elif weighting == "hour":
        w = pd.Series(1.0, index = df.index) # equal weights across hours

    W = w.sum() # = total no of markets in market weighting, = total market hours in hours weighting
    # weight of outcomes that resolve to yes
    wy = w * df["yes_outcomes"]
# --------------------------------------------------------- - -------------------------------------------------------- #

    #### CALCULATING ENTROPY H(Y)
    ## The entropy of market outcomes before we look at prices. It is just the overall uncertainty of outcomes in our sample.

    # now yes outcomes are weighted by market duration so each market contributes equally
    p_yes = wy.sum()/W
    entropy_Y = -(p_yes * np.log2(p_yes) + (1 - p_yes) * np.log2(1 - p_yes))
# --------------------------------------------------------- - -------------------------------------------------------- #

    #### CALCULATING CONDITIONAL ENTROPY H(Y|P)
    ## Now we calculate the entropy within each price bin and take the expected value to get
    ##  the uncertainty of outcomes within each price bin.

    # total weights and yes-weights in each bin
    binned = pd.DataFrame({
        "w": w.groupby(df["price_bins"], observed = True).sum(),
        "wy": wy.groupby(df["price_bins"], observed = True).sum()
    })

    # two outcomes y/n with probability p and 1-p, like a bernoulli
    binned["yes_outcomes"] = binned["wy"] / binned["w"]
    binned["entropy"] = binary_entropy(binned["yes_outcomes"])

    # get conditional entropy, i.e. expectations over the bins
    # bins use the same weights as H(Y), fixing the old problem
    binned["weights"] = binned["w"]/W
    binned["w_entropy"] = binned["weights"] * binned["entropy"]


    # weights into entropy gives expected entropy
    # summed over price bins gives entropy of outcome given price
    cond_entropy = binned["w_entropy"].sum()

    MI = entropy_Y-cond_entropy
# --------------------------------------------------------- - -------------------------------------------------------- #

    #### MILLER MADOW CORRECTION
    # Entropy estimates are biased downwards in finite samples (so MI is biased upwards), so we need to use MM correction
    # the correction term is (K-1)/2N where K = no of unique values and N = sample size
    # its ultimately really inconsequential here

    # N needs to be independent units, so we use number of markets
    # not market hours
    N = df.index.get_level_values("market_id").nunique()

    # K = price bins that have both yes and no outcomes, the only outcomes here
    # a yes outcomes share in (0,1) means the bin has both outcomes
    K = ((binned["yes_outcomes"] > 0) & (binned["yes_outcomes"] < 1)).sum()

    # (K-1)/2N is in nats, divide by log 2 to get bits
    bias = (K-1)/(2*N*np.log(2))
    MI_mm = MI - bias

    uncertainty_reduced = MI_mm * 100/entropy_Y
# --------------------------------------------------------- - -------------------------------------------------------- #
    ### Outputs

    # round causes floating point issues for the uncertainty calc
    weights_label = {"market": "Market-weighted", "hour": "Hour-weighted"}[weighting]
    hours_label = f"final {drop_last_hours} hours dropped" if drop_last_hours > 0 else "all hours"
    print(f"{weights_label}, {hours_label}, {N:,} markets")
    print(f"  {'Entropy H(Y)':<32}{entropy_Y:>9.5f} bits")
    print(f"  {'Conditional entropy H(Y|P)':<32}{cond_entropy:>9.5f} bits")
    print(f"  {'Mutual information I(Y;P)':<32}{MI:>9.5f} bits")
    print(f"  {'Miller-Madow correction':<32}{-bias:>9.5f} bits")
    print(f"  {'Corrected I(Y;P)':<32}{MI_mm:>9.5f} bits")
    print(f"  {'Uncertainty resolved by prices':<32}{uncertainty_reduced:>8.2f} %")
    print()

    return entropy_Y, cond_entropy, MI_mm, uncertainty_reduced


## 2. Point estimates

### Full sample (Table 3)

Each call prints one specification:
- **Market-weighted** (the baseline): prices resolve 38.3% of outcome uncertainty.
- **Hour-weighted**: a larger share, 43.7%, because long-lived markets resolve No more often, so they
  start with less uncertainty.
- **Final 24 hours dropped**: 34.5%, so information is not driven by trading after outcomes are known.

In [6]:
# market weighted
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel)

# hour weighted robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel, weighting="hour")

# drop last 24 hr robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel, drop_last_hours=24)

Market-weighted, all hours, 49,661 markets
  Entropy H(Y)                      0.86347 bits
  Conditional entropy H(Y|P)        0.53290 bits
  Mutual information I(Y;P)         0.33057 bits
  Miller-Madow correction          -0.00028 bits
  Corrected I(Y;P)                  0.33030 bits
  Uncertainty resolved by prices     38.25 %

Hour-weighted, all hours, 49,661 markets
  Entropy H(Y)                      0.71078 bits
  Conditional entropy H(Y|P)        0.40019 bits
  Mutual information I(Y;P)         0.31059 bits
  Miller-Madow correction          -0.00028 bits
  Corrected I(Y;P)                  0.31031 bits
  Uncertainty resolved by prices     43.66 %

Market-weighted, final 24 hours dropped, 48,872 markets
  Entropy H(Y)                      0.86102 bits
  Conditional entropy H(Y|P)        0.56402 bits
  Mutual information I(Y;P)         0.29699 bits
  Miller-Madow correction          -0.00028 bits
  Corrected I(Y;P)                  0.29671 bits
  Uncertainty resolved by prices 

### By market structure (Table 4)

Negative-risk markets belong to a set of mutually exclusive outcomes (e.g. one market per Best Picture
nominee), and arbitrage across the set can move prices without new information. We repeat the three
specifications separately for negative-risk and other markets.

Negative-risk markets resolve less uncertainty under market weighting (31.4% vs 45.3%).
It disappears under hour weighting (43.2% vs 43.8%), which points to short-lived 
markets as the source of the difference.

In [7]:
# negrisk markets
# market weighted
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == True])

# hour weighted robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == True], weighting="hour")

# drop last 24 hr robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == True], drop_last_hours=24)

Market-weighted, all hours, 30,999 markets
  Entropy H(Y)                      0.78834 bits
  Conditional entropy H(Y|P)        0.54057 bits
  Mutual information I(Y;P)         0.24777 bits
  Miller-Madow correction          -0.00044 bits
  Corrected I(Y;P)                  0.24733 bits
  Uncertainty resolved by prices     31.37 %

Hour-weighted, all hours, 30,999 markets
  Entropy H(Y)                      0.65687 bits
  Conditional entropy H(Y|P)        0.37290 bits
  Mutual information I(Y;P)         0.28397 bits
  Miller-Madow correction          -0.00044 bits
  Corrected I(Y;P)                  0.28353 bits
  Uncertainty resolved by prices     43.16 %

Market-weighted, final 24 hours dropped, 30,883 markets
  Entropy H(Y)                      0.78786 bits
  Conditional entropy H(Y|P)        0.56690 bits
  Mutual information I(Y;P)         0.22096 bits
  Miller-Madow correction          -0.00044 bits
  Corrected I(Y;P)                  0.22052 bits
  Uncertainty resolved by prices 

In [8]:
# no negrisk markets
# market weighted
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == False])

# hour weighted robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == False], weighting="hour")

# drop last 24 hr robustness
entropy_Y, cond_entropy, MI_mm, uncertainty_reduced = entropy_calculations(panel[panel["negRisk"] == False], drop_last_hours=24)

Market-weighted, all hours, 18,662 markets
  Entropy H(Y)                      0.94989 bits
  Conditional entropy H(Y|P)        0.51865 bits
  Mutual information I(Y;P)         0.43124 bits
  Miller-Madow correction          -0.00073 bits
  Corrected I(Y;P)                  0.43051 bits
  Uncertainty resolved by prices     45.32 %

Hour-weighted, all hours, 18,662 markets
  Entropy H(Y)                      0.80014 bits
  Conditional entropy H(Y|P)        0.44873 bits
  Mutual information I(Y;P)         0.35141 bits
  Miller-Madow correction          -0.00073 bits
  Corrected I(Y;P)                  0.35067 bits
  Uncertainty resolved by prices     43.83 %

Market-weighted, final 24 hours dropped, 17,989 markets
  Entropy H(Y)                      0.94852 bits
  Conditional entropy H(Y|P)        0.55744 bits
  Mutual information I(Y;P)         0.39108 bits
  Miller-Madow correction          -0.00076 bits
  Corrected I(Y;P)                  0.39032 bits
  Uncertainty resolved by prices 

## 3. Bootstrap standard errors

Standard errors come from 1,000 bootstrap draws in `bootstrap.py`. We resample markets, not
market-hours, as markets are the independent units here. 

### Full sample (standard errors in Table 3)

In [9]:
# baseline, hour weighted and drop last 24 hr
estimate, draws = bootstrap_entropy(collapse_markets(panel))
estimate, draws = bootstrap_entropy(collapse_markets(panel, weighting="hour"))
estimate, draws = bootstrap_entropy(collapse_markets(panel, drop_last_hours=24))

Bootstrap: 49,661 markets, 1,000 draws
  Entropy H(Y)                      0.86347 bits  (SE 0.00265)
  Conditional entropy H(Y|P)        0.53290 bits  (SE 0.00297)
  Corrected I(Y;P)                  0.33030 bits  (SE 0.00295)
  Uncertainty resolved by prices     38.25 %     (SE 0.31)
  95% confidence interval         [37.68%, 38.89%]

Bootstrap: 49,661 markets, 1,000 draws
  Entropy H(Y)                      0.71078 bits  (SE 0.00636)
  Conditional entropy H(Y|P)        0.40019 bits  (SE 0.00551)
  Corrected I(Y;P)                  0.31031 bits  (SE 0.00547)
  Uncertainty resolved by prices     43.66 %     (SE 0.63)
  95% confidence interval         [42.45%, 44.90%]

Bootstrap: 48,872 markets, 1,000 draws
  Entropy H(Y)                      0.86102 bits  (SE 0.00274)
  Conditional entropy H(Y|P)        0.56402 bits  (SE 0.00313)
  Corrected I(Y;P)                  0.29671 bits  (SE 0.00296)
  Uncertainty resolved by prices     34.46 %     (SE 0.32)
  95% confidence interval         [

### Negative-risk vs other markets (standard errors in Table 4)

To get a standard error for the *difference* between the two groups, both groups must use the same
bootstrap draws. `bootstrap_entropy` does this by resampling the full sample with a fixed seed and
counting only the markets in each subset. The gap's standard error is then the standard deviation of
the draw-by-draw differences, which accounts for the two estimates being computed from the same resamples.

Market-weighted, share of uncertainty resolved:

In [10]:
# negrisk vs no negrisk, same seed so both use the same draws
collapsed = collapse_markets(panel)
negrisk = panel.groupby("market_id", observed=True)["negRisk"].first().loc[collapsed[0]].to_numpy()

est_neg, draws_neg = bootstrap_entropy(collapsed, subset=negrisk)
est_no, draws_no = bootstrap_entropy(collapsed, subset=~negrisk)
gap = draws_neg["uncertainty_reduced"] - draws_no["uncertainty_reduced"]

print(f"Gap in uncertainty resolved, SE: {gap.std():.2f} percentage points")

Bootstrap: 30,999 markets, 1,000 draws
  Entropy H(Y)                      0.78834 bits  (SE 0.00412)
  Conditional entropy H(Y|P)        0.54057 bits  (SE 0.00380)
  Corrected I(Y;P)                  0.24733 bits  (SE 0.00333)
  Uncertainty resolved by prices     31.37 %     (SE 0.37)
  95% confidence interval         [30.71%, 32.12%]

Bootstrap: 18,662 markets, 1,000 draws
  Entropy H(Y)                      0.94989 bits  (SE 0.00269)
  Conditional entropy H(Y|P)        0.51865 bits  (SE 0.00481)
  Corrected I(Y;P)                  0.43051 bits  (SE 0.00497)
  Uncertainty resolved by prices     45.32 %     (SE 0.50)
  95% confidence interval         [44.39%, 46.31%]

Gap in uncertainty resolved, SE: 0.61 percentage points


Hour-weighted. The gap here is 0.7 points with a standard error of 1.3, so the two groups are
statistically indistinguishable under this weighting:

In [11]:
collapsed = collapse_markets(panel, weighting="hour")
negrisk = panel.groupby("market_id", observed=True)["negRisk"].first().loc[collapsed[0]].to_numpy()
_, draws_neg = bootstrap_entropy(collapsed, subset=negrisk)
_, draws_no = bootstrap_entropy(collapsed, subset=~negrisk)
gap = draws_neg["uncertainty_reduced"] - draws_no["uncertainty_reduced"]
print(f"Gap in uncertainty resolved, SE: {gap.std():.2f} percentage points")

Bootstrap: 30,999 markets, 1,000 draws
  Entropy H(Y)                      0.65687 bits  (SE 0.00857)
  Conditional entropy H(Y|P)        0.37290 bits  (SE 0.00687)
  Corrected I(Y;P)                  0.28353 bits  (SE 0.00697)
  Uncertainty resolved by prices     43.16 %     (SE 0.83)
  95% confidence interval         [41.60%, 44.79%]

Bootstrap: 18,662 markets, 1,000 draws
  Entropy H(Y)                      0.80014 bits  (SE 0.00927)
  Conditional entropy H(Y|P)        0.44873 bits  (SE 0.00895)
  Corrected I(Y;P)                  0.35067 bits  (SE 0.00890)
  Uncertainty resolved by prices     43.83 %     (SE 0.96)
  95% confidence interval         [42.11%, 45.71%]

Gap in uncertainty resolved, SE: 1.28 percentage points


Market-weighted again, this time for the gap in mutual information in bits
(the "Difference" row of Table 4, $I(Y;P)$ column):

In [12]:
collapsed = collapse_markets(panel)
negrisk = panel.groupby("market_id", observed=True)["negRisk"].first().loc[collapsed[0]].to_numpy()
_, draws_neg = bootstrap_entropy(collapsed, subset=negrisk)
_, draws_no = bootstrap_entropy(collapsed, subset=~negrisk)
print(f"Gap in corrected I(Y;P), SE: {(draws_neg['MI_mm'] - draws_no['MI_mm']).std():.5f} bits")

Bootstrap: 30,999 markets, 1,000 draws
  Entropy H(Y)                      0.78834 bits  (SE 0.00412)
  Conditional entropy H(Y|P)        0.54057 bits  (SE 0.00380)
  Corrected I(Y;P)                  0.24733 bits  (SE 0.00333)
  Uncertainty resolved by prices     31.37 %     (SE 0.37)
  95% confidence interval         [30.71%, 32.12%]

Bootstrap: 18,662 markets, 1,000 draws
  Entropy H(Y)                      0.94989 bits  (SE 0.00269)
  Conditional entropy H(Y|P)        0.51865 bits  (SE 0.00481)
  Corrected I(Y;P)                  0.43051 bits  (SE 0.00497)
  Uncertainty resolved by prices     45.32 %     (SE 0.50)
  95% confidence interval         [44.39%, 46.31%]

Gap in corrected I(Y;P), SE: 0.00587 bits
